# HYP-04 Phase 0 — Supply-Chain Hedge-Lag Model

> ## ⚠️ PHASE 0 · PLACEHOLDER LINKAGE · NO EDGE CLAIMED
>
> **Peers here are same-sector tickers, not a real supply-chain map** — the same
> limitation as `GST_Task2_PeerDivergence.py`. The real linkage map does not exist
> yet and is the one hard blocker on Phase 1.
>
> **Nothing in this notebook is a finding about supply-chain transmission, and
> nothing here demonstrates or is evidence for a hedging edge.** Phase 0 asks only:
> can the target be computed at all, and is the sample large enough to be worth
> continuing past this point.
>
> No real-capital logic. No live trading. No provider calls (this notebook runs
> entirely on synthetic fixtures).

---

**Authoritative documents** — this notebook does not restate them:

1. `Live Signal Engine/HYP04_SupplyChain_HedgeLag_MODEL_PLAN_v0.1_DRAFT.md`
2. `Live Signal Engine/HYP04_PLAN_ROUND2_EDITS.md` — **authoritative** where the two disagree
3. `Live Signal Engine/CLAUDE_CODE_BUILD_PROMPT_HYP04_v8.md`


## 1 · Setup

Clone the repo and install only what Phase 0 needs. `yfinance` is deliberately
absent: the only code path that uses it is the live audit runner, which makes
real provider calls and requires the repo owner's staged go-ahead. This notebook
never touches it.

In [ ]:
# If running in Colab, clone the repo. Skip if already local.
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO = 'gst-earnings-signal-research'

if IN_COLAB and not os.path.isdir(REPO):
    # Private repo: replace <TOKEN> or use the Colab secrets panel.
    # subprocess.run(['git', 'clone', f'https://<TOKEN>@github.com/bchen9996/{REPO}.git'], check=True)
    raise SystemExit('Clone the private repo first, then re-run this cell.')

if IN_COLAB:
    os.chdir(REPO)

print('cwd:', os.getcwd())


In [ ]:
%pip install --quiet 'numpy>=2.0.0' 'pandas>=2.0.0' 'scipy>=1.11.0'


In [ ]:
import sys
sys.path.insert(0, '.')   # the package is not pip-installed

import numpy as np
from hyp04_supply_chain_hedge.constants import (
    PLACEHOLDER_LINKAGE_WARNING, NO_EDGE_CLAIM, HORIZON_GRID,
)
print(PLACEHOLDER_LINKAGE_WARNING)
print()
print(NO_EDGE_CLAIM)
print()
print('horizon grid:', HORIZON_GRID[0], '...', HORIZON_GRID[-1])


## 2 · Synthetic fixtures (build step 1)

Built **first**, as their own deliverable, so the inference layer is validated
against known ground truth rather than against itself.

Peers carry their **own earnings ramp** — every company's volatility rises into
its own report regardless of any anchor. That confound is deliberate: without it,
a pipeline that skipped residualization would look identical to one that didn't.

In [ ]:
from hyp04_supply_chain_hedge.fixtures import (
    make_main_fixture, make_zero_effect_fixture,
    make_insufficient_n_fixture, make_constant_vol_fixture, summarize,
)

for builder in (make_main_fixture, make_zero_effect_fixture,
                make_insufficient_n_fixture, make_constant_vol_fixture):
    print(summarize(builder()))


## 3 · The target variable (build step 2)

$$\mathrm{RVar}(h) = \frac{1}{h}\sum_{i=1}^{h} r_i^2, \qquad
\text{target} = \log \mathrm{RVar}(h) - \log \mathrm{RVar}_{\text{ref}} - \text{baseline}$$

A **mean** squared return, not a sum, and **not** square-rooted.

### Why this matters — the bug this replaced

The rejected definition $RV(h)=\sqrt{\sum r_i^2}$ grows as $\sigma\sqrt{h}$ under
constant volatility, so `RV(h) − reference` carried a deterministic $\sqrt{h}$-shaped
trend containing **no volatility information at all** — exactly the shape a lag
profile exists to detect. The cell below demonstrates the difference is *bias* vs
*noise* by giving both definitions ~26× more data.

In [ ]:
from hyp04_supply_chain_hedge.fixtures import make_panel
from hyp04_supply_chain_hedge.target import build_feature_table, horizon_trend_diagnostic
from hyp04_supply_chain_hedge.constants import REFERENCE_WINDOW_TRADING_DAYS

def corrected_range(na, nq, npp):
    p = make_panel(f'CV_{na}_{nq}_{npp}', n_anchors=na, n_quarters=nq,
                   n_peers_per_anchor=npp, injected_lag=None, injected_effect=0.0,
                   earnings_ramp_amplitude=0.0, base_vol_range=(0.02, 0.02), seed=7)
    tr = horizon_trend_diagnostic(build_feature_table(p)['table'])
    hs = np.array(sorted(tr), dtype=float); vals = np.array([tr[h] for h in sorted(tr)])
    m = hs >= 3
    return float(vals[m].max() - vals[m].min()), p

def rejected_range(panel):
    means = {}
    for h in range(3, 21):
        acc = []
        for ev in panel.anchor_events:
            for pr in ev['peers']:
                r = panel.returns[pr]; d = ev['day']
                if d + 1 + h > r.size: continue
                rs = max(0, d - REFERENCE_WINDOW_TRADING_DAYS + 1)
                acc.append(float(np.sqrt(np.sum(r[d+1:d+1+h]**2)) - np.sqrt(np.sum(r[rs:d+1]**2))))
        if acc: means[h] = float(np.mean(acc)) / 0.02
    v = np.array(list(means.values()))
    return float(v.max() - v.min())

small_c, small_p = corrected_range(6, 6, 4)
large_c, large_p = corrected_range(24, 20, 8)
print(f'{"":22s}{"small N":>12s}{"large N (26x)":>16s}')
print(f'{"corrected log(RVar)":22s}{small_c:12.4f}{large_c:16.4f}   <- shrinks: NOISE')
print(f'{"rejected sqrt(sum r2)":22s}{rejected_range(small_p):12.4f}{rejected_range(large_p):16.4f}   <- unchanged: BIAS')


### Residualization and the two exclusion numbers

Two different quantities, kept **separate** on purpose (Patch 12) — conflating
them would hide which one actually binds:

- **retained-history fraction** — anchor-adjacent history removed from baseline *estimation*
- **bucket-population exclusion** — observations dropped because their days-to-own-earnings bucket was under-populated *once that history was in hand*

In [ ]:
built = build_feature_table(make_main_fixture())
rh, bp = built['retained_history'], built['bucket_population']
print('retained history :  min=%.3f  mean=%.3f  threshold=%.2f  peers below=%d/%d'
      % (rh['min'], rh['mean'], rh['threshold'], rh['n_peers_below_threshold'], rh['n_peers']))
print('bucket population:  excluded=%d/%d = %.1f%%  (min bucket=%d, threshold=%d)'
      % (bp['n_excluded'], bp['n_candidate_observations'], 100*bp['excluded_fraction'],
         bp['min_bucket_count_observed'], bp['min_bucket_count_threshold']))
print()
print(rh['structural_ceiling_note'])


## 4 · Local projections (build step 3)

### Onset ≠ peak — and the difference is load-bearing

With an expanding-window target, $\arg\max_h |t_h|$ lands on $L+W-1$, the **end**
of the reaction window, because the affected fraction $(h-L+1)/h$ is maximized
there. The **onset** is the first horizon clearing significance.

**Stage 2 sizes hedge tenor off the onset.** Using the peak would buy $(W-1)$ days
too much tenor and arrive after the move started.

In [ ]:
from hyp04_supply_chain_hedge.inference import local_projection_analysis

print(f'{"L":>3}{"W":>3}{"onset":>8}{"peak":>7}{"L+W-1":>8}   verdict')
for L, W in ((3, 5), (5, 5), (8, 5)):
    p = make_panel(f'LAG_{L}_{W}', n_anchors=16, n_quarters=14, n_peers_per_anchor=6,
                   injected_lag=L, injected_effect=0.55, injected_effect_window=W, seed=31)
    r = local_projection_analysis(build_feature_table(p)['table'], n_permutations=200)
    ok = 'peak==L+W-1' if r.peak_horizon == L + W - 1 else 'PEAK MISMATCH'
    print(f'{L:>3}{W:>3}{str(r.onset_horizon):>8}{str(r.peak_horizon):>7}{L+W-1:>8}   {ok}')
print()
print('Onset is LATE-BIASED, never early: at horizon L only 1/L of the averaging')
print('window is affected, so deeper lags carry thinner margin over the threshold.')


### The impulse-response profile

In [ ]:
main_table = build_feature_table(make_main_fixture())['table']
res = local_projection_analysis(main_table, n_permutations=200)

print('onset=%s  peak=%s  peak|t|=%.2f  maxT crit=%.2f  p=%.4f  significant=%s'
      % (res.onset_horizon, res.peak_horizon, res.peak_abs_t,
         res.max_t_critical_value, res.max_t_p_value, res.peak_significant))
print()
for f in res.fits:
    bar = '#' * int(min(abs(f.t_stat), 30))
    mark = ' <- onset' if f.horizon == res.onset_horizon else (' <- peak' if f.horizon == res.peak_horizon else '')
    print('h=%2d  beta=%+7.4f  t=%+7.2f  %s%s' % (f.horizon, f.beta, f.t_stat, bar, mark))


### Multiplicity: max-T permutation, never Bonferroni

$\mathrm{RVar}(h)$ is a mean over an *expanding* window, so adjacent $\beta_h$ are
correlated **by construction**. Bonferroni would be badly over-conservative against
that correlation; the permutation measures the real penalty.

Shock magnitudes are permuted **across anchor events** and re-broadcast to each
event's peers, so within-event correlation survives under the null. An iid row
shuffle would destroy it and make the null anticonservative.

In [ ]:
from hyp04_supply_chain_hedge.gate import required_n_for_k_tests

k = res.effective_independent_tests
print('effective independent tests: %.2f  (of %d horizons)' % (k, len(res.fits)))
print()
print(f'{"k":>8}{"total N required":>20}')
for kk in (1, 2, 4, k, 20):
    print(f'{kk:>8.2f}{required_n_for_k_tests(kk)["total_n_required"]:>20.0f}')
print()
print('k=1 reproduces the naive single-test baseline (126), which is a LABELLED')
print('BASELINE ONLY and never the operative gate.')


### Prewhitened CCF — secondary cross-check only

The filter is fit to the **peer-volatility** series, never the anchor shock series,
and the *same single filter* is applied to both. The cell below shows why.

In [ ]:
from hyp04_supply_chain_hedge.inference import fit_ar, prewhitened_ccf

panel = make_main_fixture()
peer = sorted(panel.returns)[0]
r2 = panel.returns[peer] ** 2
vol = np.array([float(np.mean(r2[max(0, t-5):t+1])) for t in range(r2.size)])
shock = np.zeros(panel.n_trading_days)
for ev in panel.anchor_events:
    if peer in ev['peers']:
        shock[ev['day']] = ev['shock']

print('AR(5) fit to PEER-VOL     : sum|phi| = %.4f' % np.abs(fit_ar(vol, 5)).sum())
print('AR(5) fit to ANCHOR-SHOCK : sum|phi| = %.4f  <- already near-white' % np.abs(fit_ar(shock, 5)).sum())
print()
print('=> Filtering with a shock-fit filter would be a near-no-op, leaving the')
print('   peer vol clustering -- the actual source of spurious cross-correlation --')
print('   completely untouched.')
print()
out = prewhitened_ccf(vol, shock, n_permutations=200)
print('CCF peak lag=%d  ccf=%+.4f  p=%.4f  significant=%s'
      % (out['peak_lag'], out['peak_ccf'], out['p_value'], out['significant']))
print('significance method:', out['significance_method'])
print('role:', out['role'])


## 5 · Cross-validation and the placebo test

**Leave-one-anchor-out**, never random k-fold: anchor events cluster in time and
by sector, so random folds leak information across correlated events.

The **placebo** shuffles anchor–peer assignment. If real linkage cannot be told
apart from random pairing, the model has nothing to learn regardless of algorithm.

In [ ]:
from hyp04_supply_chain_hedge.inference import leave_one_anchor_out_cv, placebo_test

zero_table = build_feature_table(make_zero_effect_fixture())['table']

cv = leave_one_anchor_out_cv(main_table, horizon=5)
print('LOOCV (MAIN, h=5): folds=%d  cv_mse=%.4f  baseline=%.4f  skill=%+.4f'
      % (cv['n_folds'], cv['cv_mse'], cv['baseline_mse_intercept_only'], cv['skill_vs_baseline']))
print('split:', cv['split'])
print()
for label, tbl in (('MAIN (real effect)', main_table), ('ZERO_EFFECT', zero_table)):
    pb = placebo_test(tbl, horizon=5, n_permutations=500)
    print('%-20s obs=%.4f  null_p95=%.4f  p=%.4f  separates=%s'
          % (label, pb['observed_abs_corr'], pb['placebo_p95'], pb['p_value'],
             pb['separates_from_random']))


## 6 · The N_eff sample-size gate (build step 3b)

$$\mathrm{DEFF} = 1 + (\bar{m}-1)\cdot\mathrm{ICC}, \qquad N_{\text{eff}} = \frac{N_{\text{pairs}}}{\mathrm{DEFF}}$$

`N_pairs` (ICC = 0) and `N_events` (ICC = 1) are the two **bounds**; neither is
gated on directly. The ICC is used exactly **once**, to place `N_eff` between them.

**The gate measures sample size, not effect size.** A zero-effect fixture with
ample N must *pass* — a well-powered null is a legitimate result.

In [ ]:
from hyp04_supply_chain_hedge.gate import evaluate_gate

K = res.effective_independent_tests
cases = [('Fixture A: zero effect, ample N', make_zero_effect_fixture, 'must PASS'),
         ('Fixture B: insufficient N, big effect', make_insufficient_n_fixture, 'must FAIL')]

for label, builder, expect in cases:
    g = evaluate_gate(build_feature_table(builder())['table'],
                      k_effective_tests=K, bootstrap_icc=False)
    r = g.report()
    print(label, ' [', expect, ']')
    print('   N_pairs=%-6d N_events=%-5d m_bar=%.2f' %
          (r['n_pairs_icc0_bound'], r['n_events_icc1_bound'], r['mean_peers_per_event']))
    print('   ICC raw=%+.4f -> floored=%.4f  (clusters=%d)' %
          (r['icc_raw_unfloored'], r['icc_used_in_deff_floored_at_0'], r['icc_n_clusters']))
    print('   DEFF=%.3f  N_eff=%.1f  target=%.0f  ->  GATE %s' %
          (r['deff'], r['n_eff'], r['target_n_fixture_derived'],
           'PASSED' if r['gate_passed'] else 'FAILED'))
    print()


## 7 · Stage 2 — hedge structure selection (build step 4)

**Rules, not a fitted model.** The sample size needed to responsibly train a
structure-choice classifier does not exist here.

**Assumed exposure:** an *existing long-equity holding* in the peer name. This is
load-bearing and stated on every recommendation — absent that holding, buying puts
is not a hedge but a directional long-volatility bet with entirely different risk
and sizing logic.

In [ ]:
from hyp04_supply_chain_hedge.stage2 import (
    Stage1Score, PeerOptionsConditions, select_hedge_structure, decision_table_summary,
)

print('DECISION TABLE (evaluated in order, first match wins)')
for row in decision_table_summary():
    print('  %-4s %-38s -> %s' % (row['rule'], row['condition'], row['action']))
print()

liquid = PeerOptionsConditions('PEER_001', 5000, 500, 0.05, 0.02, 0.35)
examples = [
    ('strong link, clean market', Stage1Score('PEER_001', 0.80, 5, 9, 2.0, True), liquid),
    ('moderate link',             Stage1Score('PEER_001', 0.35, 5, 9, 2.0, True), liquid),
    ('steep put skew',            Stage1Score('PEER_001', 0.80, 5, 9, 2.0, True),
     PeerOptionsConditions('PEER_001', 5000, 500, 0.05, 0.20, 0.35)),
    ('IV already elevated',       Stage1Score('PEER_001', 0.80, 5, 9, 2.0, True),
     PeerOptionsConditions('PEER_001', 5000, 500, 0.05, 0.02, 0.90)),
    ('lag CI too wide',           Stage1Score('PEER_001', 0.80, 5, 9, 99.0, True), liquid),
]
for label, score, cond in examples:
    rec = select_hedge_structure(score, cond)
    print('%-26s -> %-17s (%s, tenor=%s)' %
          (label, rec.structure, rec.rule_fired, rec.tenor_trading_days))
print()
print('Exposure carried on every recommendation:')
print(' ', select_hedge_structure(examples[0][1], liquid).assumed_exposure)


## 8 · What Phase 0 does and does not establish

**Established here:**
- The target variable can be computed, and its definition is free of the
  window-length artifact that would have manufactured a fake lag profile.
- The inference layer recovers a known injected lag on synthetic data, and
  correctly finds nothing when nothing is there.
- The sample-size gate responds to sample size and ignores effect size.
- The pipeline runs end-to-end and produces the expected schema.

**NOT established, and not claimed:**
- ❌ Any hedging edge, of any size, in any direction.
- ❌ Any real supply-chain transmission — the linkage here is a same-sector placeholder.
- ❌ That IV is viable as a Stage 1 target. The coverage audit measured *current*
  chains; yfinance has no historical endpoint. **Realized volatility remains the default.**

**Stated limitations:** one-way clustering only (two-way not implemented anywhere
in this repo); h = 1–2 statistically pathological; retained history structurally
capped near 0.28 for quarterly anchors; onset detection late-biased for deep lags.

---

*Nothing in this notebook constitutes investment advice, a solicitation, or a
representation that any approach is profitable.*
